<a href="https://colab.research.google.com/github/virbinet/MODIS_NDVI_Monthly_2016-2024_PierreAuger/blob/main/NDVI_GEE_MAPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Initialize Earth Engine
ee.Initialize()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# install the Proj and GEOS libraries
!apt-get install libproj-dev proj-bin
!apt-get install libgeos-dev

# install cartopy and geemap with all of the dependencies prebuilt
!pip install cartopy geemap

In [ ]:
# test some basic functionality
import ee
import geemap.foliumap as geemap
from geemap import cartoee

%pylab inline
Map = geemap.Map()

In [ ]:
pip install ephem

In [ ]:
# ==========================================
# Script to export monthly NDVI images (MODIS) 2016-2024
# ==========================================

# Import the Google Earth Engine library
import ee

# Initialize Earth Engine (make sure ee.Authenticate() was run before)
ee.Initialize()

# -----------------------------
# 1. Load the MODIS NDVI Image Collection
# -----------------------------
# MOD13Q1: MODIS NDVI product (16-day, 250 m resolution)
lst = ee.ImageCollection('MODIS/061/MOD13Q1').select('NDVI')

# -----------------------------
# 2. Define year range
# -----------------------------
start_year = 2016
end_year = 2024

# -----------------------------
# 3. Define region of interest (Bounding Box)
# -----------------------------
# Approximate coordinates
minlon, maxlon = -71, -68
minlat, maxlat = -35.8, -34.3

# Create the geometry
region = ee.Geometry.BBox(minlon, minlat, maxlon, maxlat)

# -----------------------------
# 4. Visualization parameters (optional)
# -----------------------------
# NDVI range and color palette
vis_params = {
    'min': 0,            # minimum NDVI value
    'max': 8000,         # maximum NDVI value (MODIS NDVI scaled 0-10000)
    'region': region.getInfo()['coordinates'],  # region coordinates
    'palette': ['ffffff', 'ce7e45', 'df923d', 'f1b555', 'fcd163',
                '99b718', '74a901', '66a000', '529400', '3e8601',
                '207401', '056201', '004c00', '023b01', '012e01',
                '011d01', '011301'],
    'dimensions': 800  # image size
}

# -----------------------------
# 5. Google Drive folder to save GeoTIFFs
# -----------------------------
folder_path = '/content/drive/MyDrive/Doctorado 2025/AOD Pierre Auger/NDVI Aqua/NDVI Geotiff'

# -----------------------------
# 6. Iterate over each month and export monthly mean NDVI
# -----------------------------
for month in range(1, 13):
    # Filter images for the specific month and year range
    monthly_images = lst.filter(ee.Filter.calendarRange(month, month, 'month')) \
                        .filter(ee.Filter.calendarRange(start_year, end_year, 'year'))

    # Compute monthly mean and clip to region
    monthly_mean = monthly_images.mean().clip(region)

    # Filename for each month
    filename = f"NDVI_{month:02d}_2016_2024.tif"

    try:
        # Create export task to Google Drive
        export_task = ee.batch.Export.image.toDrive(
            image=monthly_mean,
            description=f"NDVI_{month:02d}_2016_2024",
            fileNamePrefix=filename,
            folder=folder_path.split('/')[-1],  # just the folder name
            region=region,
            scale=250,          # resolution (250 m for MODIS)
            fileFormat='GeoTIFF',
            maxPixels=1e8       # pixel limit
        )

        # Start export task
        export_task.start()
        print(f"Exporting NDVI image for month {month:02d}...")

    except Exception as e:
        print(f"Error exporting NDVI image for month {month:02d}: {e}")

In [ ]:
# ==========================================
# Script to display monthly mean NDVI (MODIS) 2016-2024 in Colab
# ==========================================

import datetime
from IPython.display import Image, display
import ee

# Initialize Earth Engine (make sure ee.Authenticate() was run before)
ee.Initialize()

# -----------------------------
# 1. Load the MODIS NDVI Image Collection
# -----------------------------
# MOD13Q1: MODIS NDVI product (16-day, 250 m resolution)
lst = ee.ImageCollection('MODIS/061/MOD13Q1').select('NDVI')

# -----------------------------
# 2. Define year range
# -----------------------------
start_year = 2016
end_year = 2024

# -----------------------------
# 3. Define region of interest (Bounding Box)
# -----------------------------
# Approximate coordinates
minlon, maxlon = -71, -68
minlat, maxlat = -35.8, -34.3

# Create the geometry
region = ee.Geometry.BBox(minlon, minlat, maxlon, maxlat)

# -----------------------------
# 4. Visualization parameters
# -----------------------------
# NDVI range and color palette
vis_params = {
    'min': 0,  # minimum NDVI value
    'max': 8000,  # maximum NDVI value (MODIS NDVI scaled 0-10000)
    'region': region.getInfo()['coordinates'],  # convert geometry to coordinates
    'palette': ['ffffff', 'ce7e45', 'df923d', 'f1b555', 'fcd163',
                '99b718', '74a901', '66a000', '529400', '3e8601',
                '207401', '056201', '004c00', '023b01', '012e01',
                '011d01', '011301'],
    'dimensions': 800  # image width in pixels
}

# -----------------------------
# 5. Iterate over each month and display monthly mean NDVI
# -----------------------------
for month in range(1, 13):
    # Filter images for the specific month and year range
    monthly_images = lst.filter(ee.Filter.calendarRange(month, month, 'month')) \
                        .filter(ee.Filter.calendarRange(start_year, end_year, 'year'))

    # Compute monthly mean and clip to region
    monthly_mean = monthly_images.mean().clip(region)

    try:
        # Get thumbnail URL for visualization
        url = monthly_mean.getThumbURL(vis_params)

        # Display the image in Colab
        print(f'Monthly mean NDVI for month {month:02d} (2016-2024)')
        display(Image(url=url))  # display image directly from URL
        print('\n')

    except Exception as e:
        print(f"Error retrieving image for month {month:02d}: {e}")

In [ ]:
# ==========================================
# Script to visualize monthly NDVI maps (MODIS) 2016-2024
# with points of interest and shapefile overlay
# ==========================================

import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import rasterio
from rasterio.plot import show
import cartopy.io.shapereader as shpreader
import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import matplotlib.colors as mcolors

# -----------------------------
# 1. List of NDVI GeoTIFF files exported from MODIS
# -----------------------------
tiff_paths = [
    "/content/drive/MyDrive/Doctorado 2025/AOD Pierre Auger/NDVI Aqua/NDVI Geotiff/NDVI_01_2016_2024.tif.tif",
    "/content/drive/MyDrive/Doctorado 2025/AOD Pierre Auger/NDVI Aqua/NDVI Geotiff/NDVI_02_2016_2024.tif.tif",
    # ... continue for all 12 months
]

# -----------------------------
# 2. Month names
# -----------------------------
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# -----------------------------
# 3. Define region of interest
# -----------------------------
minlon, maxlon = -69.8, -68.8
minlat, maxlat = -35.7, -34.8

# -----------------------------
# 4. Create figure with 3 rows x 4 columns (one for each month)
# -----------------------------
fig, axes = plt.subplots(3, 4, figsize=(15, 10), subplot_kw={'projection': ccrs.PlateCarree()})
axes = axes.flatten()  # Flatten axes for easy iteration

# -----------------------------
# 5. Define custom NDVI color palette
# -----------------------------
palette = [
    '#ffffff', '#d9d2c7', '#c6bdb1', '#b5a395','#a58a7b','#a08274',
    '#99b718', '#74a901', '#66a000', '#529400', '#3e8601', '#207401',
    '#056201', '#004c00', '#023b01', '#012e01', '#011d01', '#011301',
    '#010b01', '#000801'
]
cmap = mcolors.ListedColormap(palette)

# -----------------------------
# 6. Plot each NDVI GeoTIFF
# -----------------------------
for i, tiff_path in enumerate(tiff_paths):
    with rasterio.open(tiff_path) as dataset:
        data = dataset.read(1) * 0.0001  # scale NDVI values (MODIS scale)
        transform = dataset.transform

    ax = axes[i]
    ax.set_extent([minlon, maxlon, minlat, maxlat], crs=ccrs.PlateCarree())

    # Plot NDVI raster
    show(data, transform=transform, cmap=cmap, ax=ax, vmin=0, vmax=1)

    # -----------------------------
    # 6a. Overlay shapefile (contour of Pierre Auger Observatory)
    # -----------------------------
    shapefile_path = "/content/drive/MyDrive/Doctorado 2025/AOD Pierre Auger/contornopierreauger.shp/contornoauger.shp"
    shape_reader = shpreader.Reader(shapefile_path)
    ax.add_geometries(shape_reader.geometries(), ccrs.PlateCarree(),
                      edgecolor="black", facecolor="none")

    # -----------------------------
    # 6b. Add points of interest (stations)
    # -----------------------------
    sites = {
        "CLF": (-69.33684, -35.280303),
        "CO": (-69.5986467, -35.1140686),
        "AERA": (-69.5332147, -35.1086822),
        "XLF": (-69.2871905, -35.1865919),
        "LA": (-69.2102577, -34.9355132),
        "LL": (-69.4455194, -35.4960829),
        "LM": (-69.01117, -35.292392),
    }

    for name, (lon, lat) in sites.items():
        ax.scatter(lon, lat, color='k', marker='*', zorder=2, s=3)
        if name == "CO":
            ax.text(lon, lat, f' {name}', color='k', fontsize=9,
                    fontweight='bold', ha='right', va='top')
        else:
            ax.text(lon, lat, f' {name}', color='k', fontsize=9,
                    fontweight='bold', ha='left', va='top')

    # -----------------------------
    # 6c. Add gridlines
    # -----------------------------
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                      linewidth=0.5, color='k', alpha=0.5, linestyle=':')
    gl.ylocator = mticker.FixedLocator(np.arange(-90, 90, 0.3))
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 0, 0.4))
    gl.right_labels = False
    gl.ylabels_left = True
    gl.yformatter = LATITUDE_FORMATTER
    gl.top_labels = False
    gl.xlabels_bottom = True
    gl.xformatter = LONGITUDE_FORMATTER
    gl.ylabel_style = {'size': 7, 'rotation': 90, 'ha': 'left'}
    gl.xlabel_style = {'size': 7}

    # Add month title
    ax.set_title(months[i], fontsize=10)

# -----------------------------
# 7. Add a single colorbar
# -----------------------------
cbar = plt.colorbar(axes[0].images[0], ax=axes, orientation='vertical',
                    label="NDVI", fraction=0.01, pad=0.01)
cbar.set_ticks(np.arange(0, 1.1, 0.1))

# -----------------------------
# 8. Overall figure title
# -----------------------------
fig.suptitle('Monthly NDVI (MODIS) (2016-2024)')

# -----------------------------
# 9. Save figure
# -----------------------------
output_path = "/content/drive/MyDrive/Doctorado 2025/AOD Pierre Auger/NDVI Aqua/NDVI_Monthly.png"
plt.savefig(output_path, dpi=400, bbox_inches='tight')

# -----------------------------
# 10. Display figure
# -----------------------------
plt.show()